# Setup

This notebook is designed to guide you through the setup of this repository. First, ensure the code is fully loaded.

**NOTE**: Do not edit any of these cells unless they begin with a TODO comment.


In [ ]:
%load_ext autoreload
%autoreload 2

from ravelry_scraper.constants import DATA_DIR

%pip install -e {PROJECT_ROOT}

## Ravelry Scraping

Create a Ravelry account and procure credentials for the API by following the instructions [here](https://www.ravelry.com/api). If you would like to jump start the analysis, reach out to the repository owner for a link to pre-scraped data. Store these values in the cell below.


In [ ]:
# TODO: Update these values with your actual Ravelry API credentials and the URL for the scraped data
RAVELRY_USER = "your-ravelry-api-username"
RAVELRY_PASSWORD = "your-ravelry-api-password"
SCRAPED_DATA_URL = "google-drive-url"


Create the `.env` file.

In [ ]:
from pathlib import Path

from dotenv import load_dotenv, set_key

env_file = Path(".env")

DEFAULT_RAVELRY_USER = "your-ravelry-api-username"
DEFAULT_RAVELRY_PASSWORD = "your-ravelry-api-password"
DEFAULT_SCRAPED_DATA_URL = "google-drive-url"

if not env_file.is_file():
    env_file.touch()
    if RAVELRY_USER != DEFAULT_RAVELRY_USER:
        set_key(env_file, "RAVELRY_USER", RAVELRY_USER)
    if RAVELRY_PASSWORD != DEFAULT_RAVELRY_PASSWORD:
        set_key(env_file, "RAVELRY_PASSWORD", RAVELRY_PASSWORD)
    if SCRAPED_DATA_URL != DEFAULT_SCRAPED_DATA_URL:
        set_key(env_file, "SCRAPED_DATA_URL", SCRAPED_DATA_URL)

load_dotenv(env_file, override=True)

Verify the environment variables were set successfully:

In [ ]:
import os

if (os.getenv("RAVELRY_USER") is None or
    os.getenv("RAVELRY_USER") == DEFAULT_RAVELRY_USER):
    raise OSError("RAVELRY_USER must be set in the environment.")
else:
    print("Successfully set RAVELRY_USER")

if (os.getenv("RAVELRY_PASSWORD") is None or
    os.getenv("RAVELRY_PASSWORD") == DEFAULT_RAVELRY_PASSWORD):
    raise OSError("RAVELRY_PASSWORD must be set in the environment.")
else:
    print("Successfully set RAVELRY_PASSWORD")

if (os.getenv("SCRAPED_DATA_URL") is None or
    os.getenv("SCRAPED_DATA_URL") == DEFAULT_SCRAPED_DATA_URL):
    raise OSError("SCRAPED_DATA_URL must be set in the environment.")
else:
    print("Successfully set SCRAPED_DATA_URL")

### Load Existing Data

Download scraped data from Google Drive:

In [ ]:
import os
from pathlib import Path

import gdown

output_dir = DATA_DIR / "raw" / "downloaded"
output_dir.mkdir(parents=True, exist_ok=True)

gdown.download_folder(os.getenv("SCRAPED_DATA_URL"), output=str(output_dir), quiet=False)


Verify that the download worked by printing the files to terminal.

In [ ]:
downloaded_files = [f for f in Path(output_dir).rglob("*") if f.is_file()]

for f in sorted(downloaded_files):
    size_mb = f.stat().st_size / (1024 * 1024)
    print(f" - {f.relative_to(output_dir)} ({size_mb:.2f} MB)")


You should see files for Patterns, Projects, Designers, and Yarns. Copy these files to the main directory, `f"{DATA_DIR}/raw"`.

In [ ]:
import shutil

from ravelry_scraper.helpers.parquet import combine_parquet_files, de_dup_parquet

for downloaded_file in sorted(downloaded_files):
    existing_file = DATA_DIR / "raw" / downloaded_file.name
    if existing_file.exists():
        output_file = combine_parquet_files(
            files=[existing_file, downloaded_file],
            destination_path=existing_file
        )
        de_dup_parquet(output_file)
    else:
        shutil.move(downloaded_file, existing_file)


Verify that you see these files. They should be within the root directory of `cis-5450`, NOT in `cis-5450/notebooks`.
- `data/raw/patterns.parquet`
- `data/raw/projects.parquet`
- `data/raw/designers.parquet`
- `data/raw/yarns.parquet`

If all looks good, you can move on to the next notebook for Data Wrangling.